# Head Writing Direction Analysis

What directions does each attention head write into the residual stream?
SVD-based analysis of writing directions, unembed alignment,
consistency across positions, and magnitude contributions.

## Why This Matters

Attention head writing direction reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_writing_direction_analysis import (
    head_writing_directions, head_unembed_alignment,
    head_direction_consistency, head_writing_magnitude,
    head_writing_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Writing Directions via SVD

Each head writes into the residual stream through W_O. SVD reveals
the principal writing directions and effective dimensionality.

In [ ]:
for layer in range(4):
    result = head_writing_directions(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_rank={result['mean_rank']:.2f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: rank={h['effective_rank']:.2f}, "
              f"top_sv={h['top_sv']:.4f}, conc={h['sv_concentration']:.4f}")

## Unembed Alignment

Which vocabulary items does each head's mean output promote?

In [ ]:
for layer in range(4):
    result = head_unembed_alignment(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}:")
    for h in result['per_head']:
        print(f"    H{h['head']}: top tokens={h['top_tokens'][:3]}, "
              f"logits={[f'{l:.3f}' for l in h['top_logits'][:3]]}")

## Direction Consistency

Does each head write in the same direction regardless of position?

In [ ]:
for layer in range(4):
    result = head_direction_consistency(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_consistent']}/{len(result['per_head'])} consistent")
    for h in result['per_head']:
        flag = ' [CONSISTENT]' if h['is_consistent'] else ''
        print(f"    H{h['head']}: consistency={h['direction_consistency']:.4f}{flag}")

## Writing Magnitude

How much does each head contribute to the residual stream?

In [ ]:
for layer in range(4):
    result = head_writing_magnitude(model, tokens, layer=layer)
    print(f"  Layer {layer}: dominant=H{result['dominant_head']}")
    for h in result['per_head']:
        bar = '█' * int(h['fraction'] * 40)
        print(f"    H{h['head']}: norm={h['mean_norm']:.4f}, "
              f"frac={h['fraction']:.3f} {bar}")

## Writing Summary

In [ ]:
result = head_writing_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean_rank={p['mean_rank']:.2f}, "
          f"consistent={p['n_consistent']}")